# Creation of $V_{eval}$
## Retrieval of Raw Data

The starting point for constructing $V_{eval}$ is the *ClapNQ* dataset (Rosenthal et al., 2024). The dataset contains 300 questions and is available at:

```bash
data/wikipedia/clapnq_baseline/clapnq_dev_answerable_orig.jsonl
```

All entries in the JSONL file follow the same structure and contain, among other fields, a `document_url` field. This field points to the Wikipedia article on which the respective question is based. The URL also contains an `oldid` parameter, which identifies a specific revision of the corresponding Wikipedia article. Using this revision identifier, the exact version of the Wikipedia article underlying each question can be retrieved, allowing the original data basis of ClapNQ to be reconstructed.

## Cleaning

The raw Wikipedia articles contain substantial structural and markup-related noise that is not relevant for question generation. In particular, the articles contain nested MediaWiki syntax that needs to be removed before further processing.

For this purpose, `mwparserfromhell` is used to parse and clean the MediaWiki markup. The resulting representation contains the textual content of the articles organized by sections. In addition, a blacklist is applied to remove sections that do not provide relevant informational content for our use case. These include:

```bash
references
notes
notes and references
footnotes
citations
works cited
further reading
bibliography
sources
external links
see also
related articles
gallery
```

The cleaning step therefore produces a simplified representation of each Wikipedia article that retains the relevant textual content while removing references, navigation elements, and other auxiliary sections.

## Chunking

The resulting corpus is segmented using recursive, character-based chunking with a fixed chunk size of 3,500 characters and an overlap of 525 characters. The complete article text is split independently of the original section boundaries.

After chunking, the section affiliation of each chunk $d_i$ is reconstructed using an index mapping based on the positions of the original section texts. This allows each chunk to be associated with its corresponding Wikipedia section even though the chunking itself is performed over the complete article text.

Each chunk is enriched with two types of metadata:

* **Descriptive metadata:** A document header containing the Wikipedia article title and the corresponding section title is prepended to the chunk text. This preserves the original document context after segmentation.

* **Structural metadata:** For deterministic context expansion, ChromaDB stores the document identifier $P(d_i)$, the sequential position of the chunk $Idx(d_i)$, and the total number of chunks $N(d_i)$. These metadata fields allow neighboring chunks to be identified deterministically and retrieved as additional context when required.

In [4]:
import json
import os
import json
import requests
import html
import re
import mwparserfromhell
from urllib.parse import urlparse, parse_qs

file_path = "../data/wikipedia/clapnq_baseline/clapnq_dev_answerable_orig.jsonl"
urls = []

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        data = json.loads(line)
        urls.append(data["document_url"])

OUTPUT_DIR = "../data/wikipedia_articles"

def extract_oldid(url):
    """Extrahiert die oldid aus der URL."""
    decoded_url = html.unescape(url)
    parsed_url = urlparse(decoded_url)
    query_params = parse_qs(parsed_url.query)
    
    if 'oldid' in query_params:
        return query_params['oldid'][0]
    return None

def fetch_and_clean_article(oldid):
    api_url = "https://en.wikipedia.org/w/api.php"
    
    headers = {
        "User-Agent": "WikipediaQABuilder/1.0 (firstname.lastname@youremail.com) - Python Script"
    }
    
    params = {
        "action": "query",
        "prop": "revisions",
        "rvprop": "content",
        "revids": oldid,
        "format": "json",
        "rvslots": "main"
    }

    response = requests.get(api_url, params=params, headers=headers)
    response.raise_for_status()
    data = response.json()

    pages = data.get("query", {}).get("pages", {})

    page = list(pages.values())[0]
    title = page.get("title", "Unknown_Title")
    
    wikitext = page["revisions"][0]["slots"]["main"]["*"]

    wikicode = mwparserfromhell.parse(wikitext)

    # remove all <ref> Tags und HTML-Comments
    for tag in wikicode.filter_tags():
        if tag.tag.lower() == "ref":
            try:
                wikicode.remove(tag)
            except ValueError:
                pass
                
    for comment in wikicode.filter_comments():
        try:
            wikicode.remove(comment)
        except ValueError:
            pass

    sections_data = []
    
    # split Article into Sections
    for section in wikicode.get_sections(include_lead=True, flat=True):
        
        headings = section.filter_headings()
        if headings:
            heading = headings[0]
            section_title = str(heading.title).strip()
            level = heading.level
            section.remove(heading)
        else:
            section_title = "Introduction"
            level = 1

        clean_text = section.strip_code().strip()
        
        clean_text = re.sub(r'\n{3,}', '\n\n', clean_text)

        if clean_text:
            sections_data.append({
                "section_title": section_title,
                "level": level,
                "content": clean_text
            })

    return {
        "title": title,
        "oldid": oldid,
        "url": f"https://en.wikipedia.org/w/index.php?title={title.replace(' ', '_')}&oldid={oldid}",
        "sections": sections_data
    }

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

existing_files = os.listdir(OUTPUT_DIR)

for url in urls:
    oldid = extract_oldid(url)
    structured_data = fetch_and_clean_article(oldid)
    
    if structured_data:
        safe_title = structured_data["title"].replace(" ", "_").replace("/", "-")
        filename = f"{safe_title}_{oldid}.json"
        filepath = os.path.join(OUTPUT_DIR, filename)
        
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(structured_data, f, indent=4, ensure_ascii=False)
            
        print(f"Saved: {filepath}")
        
        existing_files.append(filename)

## Cleaning the wikipedia articles - removing redundant sections

In [ ]:
import os
import json

INPUT_DIR = "../data/wikipedia_articles"
OUTPUT_DIR = "../data/wikipedia_articles_cleaned"

def is_junk_section(title):

    clean_title = title.lower().strip()
    
    junk_exact = {
        "references", "notes", "notes and references", "footnotes", 
        "citations", "works cited", "further reading", "bibliography", 
        "sources", "external links", "see also", "related articles",
        "gallery"
    }
    
    if clean_title in junk_exact:
        return True
        
    if "external link" in clean_title or "further reading" in clean_title:
        return True
        
    return False


os.makedirs(OUTPUT_DIR, exist_ok=True)
processed_count = 0

for filename in os.listdir(INPUT_DIR):
    if not filename.endswith(".json"):
        continue

    input_path = os.path.join(INPUT_DIR, filename)
    output_path = os.path.join(OUTPUT_DIR, filename)

    with open(input_path, "r", encoding="utf-8") as f:
        try:
            article_data = json.load(f)
        except json.JSONDecodeError:
            print(f"Error reading file {filename} (No valid JSON) - Skipping.")
            continue

    original_sections = article_data.get("sections", [])
    cleaned_sections = []
    removed_titles = []

    for sec in original_sections:
        title = sec.get("section_title", "")
        if not is_junk_section(title):
            cleaned_sections.append(sec)
        else:
            removed_titles.append(title)

    article_data["sections"] = cleaned_sections

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(article_data, f, indent=4, ensure_ascii=False)

    processed_count += 1

### Chunking

* chunk_size = 3500
* overlap = 525

In [ ]:
import json
import os
import shutil
import time

import chromadb
from dotenv import load_dotenv
from google import genai
from google.genai import types
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm


# Configuration
load_dotenv(override=True)

INPUT_DIR = "../data/wikipedia_articles_cleaned"
DB_PATH = "./chroma_db_wiki"
COLLECTION_NAME = "wikipedia_eval_chunks"

CHUNK_SIZE = 3500
CHUNK_OVERLAP = 525
EMBEDDING_BATCH_SIZE = 100
EMBEDDING_MODEL = "gemini-embedding-001"


client = genai.Client()


def reset_database():
    """Remove and recreate the ChromaDB database."""
    if os.path.exists(DB_PATH):
        shutil.rmtree(DB_PATH)

    chroma_client = chromadb.PersistentClient(path=DB_PATH)
    return chroma_client.get_or_create_collection(name=COLLECTION_NAME)


def create_text_splitter():
    """Create the text splitter used for document chunking."""
    return RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", r"(?<=\. )", " ", ""],
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        is_separator_regex=True,
    )


def get_embeddings(texts):
    """Generate embeddings in batches and retry rate-limited requests."""
    embeddings = []

    for start in range(0, len(texts), EMBEDDING_BATCH_SIZE):
        batch = texts[start : start + EMBEDDING_BATCH_SIZE]

        while True:
            try:
                response = client.models.embed_content(
                    model=EMBEDDING_MODEL,
                    contents=batch,
                    config=types.EmbedContentConfig(
                        task_type="RETRIEVAL_DOCUMENT"
                    ),
                )
                embeddings.extend(
                    embedding.values for embedding in response.embeddings
                )
                break

            except Exception as error:
                if "429" not in str(error):
                    raise

                time.sleep(5)

    return embeddings


def build_article_text(article):
    """Combine article sections and record their positions."""
    full_text = ""
    section_map = []

    for section in article.get("sections", []):
        title = section.get("section_title", "Unknown")
        content = section.get("content", "")

        if not content:
            continue

        start = len(full_text)
        full_text += content + "\n\n"
        end = len(full_text)

        section_map.append((start, end, title))

    return full_text, section_map


def get_chunks(full_text, section_map, article_title, splitter):
    """Split an article and assign each chunk to its corresponding sections."""
    chunks = []
    search_position = 0

    for chunk in splitter.split_text(full_text):
        start = full_text.find(chunk, search_position)

        if start == -1:
            start = search_position

        end = start + len(chunk)
        search_position = start + 1

        sections = [
            title
            for section_start, section_end, title in section_map
            if max(start, section_start) < min(end, section_end)
        ]

        section_title = ", ".join(dict.fromkeys(sections)) or "Unknown"

        chunks.append(
            f"Title: {article_title}\n"
            f"Section(s): {section_title}\n\n"
            f"{chunk}"
        )

    return chunks


def process_article(article, filename, collection, splitter):
    """Process and store a single Wikipedia article."""
    article_title = article.get("title", "Unknown")
    parent_id = str(article.get("oldid", filename))

    full_text, section_map = build_article_text(article)
    chunks = get_chunks(
        full_text,
        section_map,
        article_title,
        splitter,
    )

    if not chunks:
        return

    total_chunks = len(chunks)

    ids = [
        f"{parent_id}#chunk_{index}"
        for index in range(total_chunks)
    ]

    metadata = [
        {
            "parent_id": parent_id,
            "chunk_index": index,
            "total_chunks": total_chunks,
            "article_title": article_title,
        }
        for index in range(total_chunks)
    ]

    embeddings = get_embeddings(chunks)

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=chunks,
        metadatas=metadata,
    )


if not os.path.isdir(INPUT_DIR):
    raise FileNotFoundError(
        f"Input directory not found: {INPUT_DIR}"
    )

collection = reset_database()
splitter = create_text_splitter()

files = sorted(
    filename
    for filename in os.listdir(INPUT_DIR)
    if filename.endswith(".json")
)

for filename in tqdm(files, desc="Processing articles"):
    filepath = os.path.join(INPUT_DIR, filename)

    with open(filepath, "r", encoding="utf-8") as file:
        article = json.load(file)

    try:
        process_article(
            article,
            filename,
            collection,
            splitter,
        )
    except Exception as error:
        print(f"Failed to process {filename}: {error}")

You can now connect to the database as follows:

```bash
import random
import chromadb

DB_PATH = "./chroma_db_wiki"

chroma_client = chromadb.PersistentClient(path=DB_PATH)
vector_collection = chroma_client.get_or_create_collection(
    name="wikipedia_eval_chunks"
)
```